# RAPiDock Colab Runner for GHS-R Peptides

This notebook runs RAPiDock in Google Colab (Linux) and exports peptide docking outputs for your local project.

Workflow:
1. Clone RAPiDock and install dependencies.
2. Download `rapidock_local.pt` checkpoint.
3. Upload your receptor PDB and peptide CSV.
4. Run either a single smoke test or full batch.
5. Zip outputs and download them.

After download, place outputs into your local repo as `outputs_rapidock/`, then run locally:

```bash
python scripts/parse_rapidock_results.py
python scripts/parse_results.py
python scripts/visualize.py
```

In [ ]:
#@title 1) Clone RAPiDock
!git clone https://github.com/huifengzhao/RAPiDock.git
%cd /content/RAPiDock

In [ ]:
#@title 2) Install dependencies (CPU-safe defaults)
!pip install -U pip
!pip install pyyaml pandas biopython MDAnalysis rdkit-pypi fair-esm e3nn pyrosetta-installer

# RAPiDock uses torch geometric extensions; these wheels are CPU-compatible for Colab smoke tests.
!pip install torch-geometric torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.4.0+cpu.html

In [ ]:
#@title 3) Download RAPiDock local checkpoint
!mkdir -p /content/RAPiDock/train_models/CGTensorProductEquivariantModel
!wget -O /content/RAPiDock/train_models/CGTensorProductEquivariantModel/rapidock_local.pt "https://zenodo.org/records/14193621/files/rapidock_local.pt?download=1"
!ls -lh /content/RAPiDock/train_models/CGTensorProductEquivariantModel/rapidock_local.pt

In [ ]:
#@title 4) Upload inputs from your local project
# Upload these files from local repo:
# - data/ghsr_receptor.pdb  (or data/ghsr_pocket.pdb)
# - inputs_rapidock/protein_peptide.csv (for full-batch mode)
#
# Optional for single-case mode: only receptor PDB is needed.
from google.colab import files
uploaded = files.upload()
print('Uploaded files:', list(uploaded.keys()))

In [ ]:
#@title 5) Set paths and quick sanity check
import os

RECEPTOR_PDB = '/content/ghsr_receptor.pdb'  # Change if uploaded with different name
BATCH_CSV = '/content/protein_peptide.csv'   # Change if uploaded with different name
OUTPUT_SINGLE = '/content/outputs_rapidock_test'
OUTPUT_BATCH = '/content/outputs_rapidock'

print('RECEPTOR_PDB exists:', os.path.exists(RECEPTOR_PDB))
print('BATCH_CSV exists:', os.path.exists(BATCH_CSV))
print('RAPiDock checkpoint exists:', os.path.exists('/content/RAPiDock/train_models/CGTensorProductEquivariantModel/rapidock_local.pt'))

In [ ]:
#@title 6) Single-case smoke test (hexarelin)
!python /content/RAPiDock/inference.py \
  --complex_name hexarelin_test \
  --protein_description "$RECEPTOR_PDB" \
  --peptide_description "H[DTR]AW[DPN]K" \
  --output_dir "$OUTPUT_SINGLE" \
  --model_dir /content/RAPiDock/train_models/CGTensorProductEquivariantModel \
  --ckpt rapidock_local.pt \
  --scoring_function ref2015 \
  --N 1 --batch_size 1 \
  --no_final_step_noise \
  --inference_steps 8 --actual_steps 8 \
  --conformation_partial 1:1:1 \
  --cpu 4

In [ ]:
#@title 7) Verify single-case output artifacts
!ls -R "$OUTPUT_SINGLE"

# Expected key file:
# /content/outputs_rapidock_test/hexarelin_test/ref2015_score.csv

In [ ]:
#@title 8) Full peptide batch run (uses uploaded protein_peptide.csv)
# Skip this cell if you only need single-case smoke test.
!python /content/RAPiDock/inference.py \
  --protein_peptide_csv "$BATCH_CSV" \
  --output_dir "$OUTPUT_BATCH" \
  --model_dir /content/RAPiDock/train_models/CGTensorProductEquivariantModel \
  --ckpt rapidock_local.pt \
  --scoring_function ref2015 \
  --N 10 --batch_size 4 \
  --no_final_step_noise \
  --inference_steps 16 --actual_steps 16 \
  --conformation_partial 1:1:1 \
  --cpu 8

In [ ]:
#@title 9) Zip and download RAPiDock outputs
from google.colab import files

# Choose which output folder to export:
EXPORT_DIR = OUTPUT_BATCH if os.path.exists(OUTPUT_BATCH) else OUTPUT_SINGLE
EXPORT_ZIP = '/content/outputs_rapidock_colab.zip'

!zip -r "$EXPORT_ZIP" "$EXPORT_DIR"
files.download(EXPORT_ZIP)

## Local follow-up after download

1. Unzip into your repo so outputs land under `outputs_rapidock/`.
2. Run locally:

```bash
python scripts/parse_rapidock_results.py
python scripts/parse_results.py
python scripts/visualize.py
```

This merges RAPiDock peptide rows with Boltz small-molecule rows in `results/ranking.csv`.